In [ ]:
# In[1]:

import pandas as pd
import pytz
import os

tz = pytz.timezone('Asia/Shanghai')

# Filenames
filenames = ["metric.csv", "trace.csv", "log.csv", "error_logs.csv"]

# Helper to load CSV if exists
def load_csv(fn):
    if os.path.exists(fn):
        return pd.read_csv(fn)
    else:
        return None

# Reuse variables for loaded dataframes
metric = load_csv("metric.csv")
trace = load_csv("trace.csv")
log = load_csv("log.csv")
error_logs = load_csv("error_logs.csv")

dfs = {
    "metric.csv": metric,
    "trace.csv": trace,
    "log.csv": log,
    "error_logs.csv": error_logs,
}

def ts_to_shanghai(ts):
    # ts is epoch seconds (int/float) or NaN
    if pd.isna(ts):
        return pd.NA
    try:
        return pd.to_datetime(int(ts), unit="s", utc=True).tz_convert(tz).strftime("%Y-%m-%d %H:%M:%S %Z")
    except Exception:
        return pd.NA

def first_n_summary(arr, n=50):
    if arr is None:
        return ""
    arr = list(map(str, arr))
    total = len(arr)
    if total == 0:
        return ""
    sample = arr[:n]
    s = ", ".join(sample)
    if total > n:
        s += f", ...(+{total-n})"
    return s

summary_rows = []
for fname, df in dfs.items():
    if df is None:
        summary_rows.append({
            "file": fname,
            "row_count": 0,
            "min_timestamp": pd.NA,
            "max_timestamp": pd.NA,
            "unique_cmdb_ids_sample": "",
            "unique_name_field_sample": "",
            "columns": "",
        })
        continue

    # Basic counts and columns
    row_count = len(df)
    cols = list(df.columns)

    # Timestamps
    if "timestamp" in df.columns and df["timestamp"].notna().any():
        min_ts_val = df["timestamp"].min()
        max_ts_val = df["timestamp"].max()
        min_ts = ts_to_shanghai(min_ts_val)
        max_ts = ts_to_shanghai(max_ts_val)
    else:
        min_ts = pd.NA
        max_ts = pd.NA

    # cmdb_id uniques sample (limit 50)
    if "cmdb_id" in df.columns:
        unique_cmdb = df["cmdb_id"].dropna().unique()
        unique_cmdb_s = first_n_summary(unique_cmdb, 50)
    else:
        unique_cmdb_s = ""

    # name field depending on file
    name_field = ""
    name_sample = ""
    if fname == "metric.csv" and "kpi_name" in df.columns:
        name_field = "kpi_name"
        unique_names = df["kpi_name"].dropna().unique()
        name_sample = first_n_summary(unique_names, 50)
    elif fname == "trace.csv" and "trace_name" in df.columns:
        name_field = "trace_name"
        unique_names = df["trace_name"].dropna().unique()
        name_sample = first_n_summary(unique_names, 50)
    elif fname == "log.csv" and "log_name" in df.columns:
        name_field = "log_name"
        unique_names = df["log_name"].dropna().unique()
        name_sample = first_n_summary(unique_names, 50)
    else:
        # for error_logs.csv or other cases: list none but include file's columns (per instruction)
        name_field = ""
        name_sample = ""

    summary_rows.append({
        "file": fname,
        "row_count": row_count,
        "min_timestamp": min_ts,
        "max_timestamp": max_ts,
        "unique_cmdb_ids_sample": unique_cmdb_s,
        "name_field": name_field,
        "unique_name_field_sample": name_sample,
        "columns": ", ".join(cols),
    })

summary_df = pd.DataFrame(summary_rows, columns=[
    "file", "row_count", "min_timestamp", "max_timestamp",
    "unique_cmdb_ids_sample", "name_field", "unique_name_field_sample", "columns"
])

# Keep output compact: show up to first 20 rows (we have 4)
summary_df

```
Out[1]:
```
Summary (timestamps shown in Asia/Shanghai, UTC+8):

- metric.csv
  - Rows: 319,721
  - Time range: 2022-03-20 13:30:00 CST to 2022-03-20 14:30:00 CST
  - Sample unique cmdb_id (first ~50): adservice, adservice-0, adservice-1, adservice...
  - Name field: kpi_name
  - Sample unique kpi_name (first ~50): app.grpc.count, app.grpc.mrt, app.grpc.rr, app...
  - Columns: timestamp, cmdb_id, kpi_name, value

- trace.csv
  - Rows: 35,904
  - Time range: 2022-03-20 14:00:00 CST to 2022-03-20 14:29:00 CST
  - Sample unique cmdb_id (first ~50): adservice-0, adservice-1, adservice-2, adservi...
  - Name field: trace_name
  - Sample unique trace_name (first ~50): trace.from_frontend-0.duration_mean, trace.fro...
  - Columns: timestamp, cmdb_id, trace_name, value

- log.csv
  - Rows: 1,850
  - Time range: 2022-03-20 14:00:00 CST to 2022-03-20 14:30:00 CST
  - Sample unique cmdb_id (first ~50): adservice-0, adservice-1, adservice-2, cartser...
  - Name field: log_name
  - Sample unique log_name (first ~50): log.error_count, log.row_count
  - Columns: timestamp, cmdb_id, log_name, value

- error_logs.csv
  - Rows: 0 (empty)
  - Time range: none
  - Unique cmdb_id: none
  - Name field: n/a (file columns: timestamp, cmdb_id, message)
  - Columns: timestamp, cmdb_id, message

If you want, I can expand any of the samples (show more unique cmdb_id or KPI/trace/log names up to 50) or provide counts by cmdb_id for a file.

The original code execution output of IPython Kernel is also provided below for reference:

file  row_count            min_timestamp            max_timestamp                             unique_cmdb_ids_sample  name_field                           unique_name_field_sample                                columns
0      metric.csv     319721  2022-03-20 13:30:00 CST  2022-03-20 14:30:00 CST  adservice, adservice-0, adservice-1, adservice...    kpi_name  app.grpc.count, app.grpc.mrt, app.grpc.rr, app...    timestamp, cmdb_id, kpi_name, value
1       trace.csv      35904  2022-03-20 14:00:00 CST  2022-03-20 14:29:00 CST  adservice-0, adservice-1, adservice-2, adservi...  trace_name  trace.from_frontend-0.duration_mean, trace.fro...  timestamp, cmdb_id, trace_name, value
2         log.csv       1850  2022-03-20 14:00:00 CST  2022-03-20 14:30:00 CST  adservice-0, adservice-1, adservice-2, cartser...    log_name                     log.error_count, log.row_count    timestamp, cmdb_id, log_name, value
3  error_logs.csv          0                     <NA>                     <NA>                                                                                                                              timestamp, cmdb_id, message```
```

In [ ]:
# In[2]:

import pandas as pd
import os

# Reuse previously loaded variables if present, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv") if os.path.exists("metric.csv") else None
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv") if os.path.exists("trace.csv") else None
try:
    log
except NameError:
    log = pd.read_csv("log.csv") if os.path.exists("log.csv") else None

# Helper to compute top N cmdb_id counts and unique list (limited)
def cmdb_stats(df, top_n=20, unique_limit=200):
    if df is None or "cmdb_id" not in df.columns:
        top = pd.DataFrame(columns=["cmdb_id", "count"])
        unique_limited = []
        unique_full = []
    else:
        counts = df["cmdb_id"].dropna().value_counts()
        top = counts.head(top_n).reset_index().rename(columns={"index": "cmdb_id", "cmdb_id": "count"})
        unique_full = list(counts.index.tolist())
        unique_limited = unique_full[:unique_limit]
    return top, unique_limited, unique_full

# Compute per-file stats
metric_top20_cmdb, metric_unique_cmdb_limited, metric_unique_cmdb_full = cmdb_stats(metric, 20, 200)
trace_top20_cmdb, trace_unique_cmdb_limited, trace_unique_cmdb_full = cmdb_stats(trace, 20, 200)
log_top20_cmdb, log_unique_cmdb_limited, log_unique_cmdb_full = cmdb_stats(log, 20, 200)

# Combined observed cmdb_id set (union of the three files)
combined_obs_set = set(metric_unique_cmdb_full) | set(trace_unique_cmdb_full) | set(log_unique_cmdb_full)

# Provided possible root-cause components list
possible_roots = [
    "node-1","node-2","node-3","node-4","node-5","node-6",
    "frontend-0","frontend-1","frontend-2","frontend2-0",
    "shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
    "checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
    "currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
    "adservice-0","adservice-1","adservice-2","adservice2-0",
    "emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
    "cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
    "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
    "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
    "paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0",
    "frontend","shippingservice","checkoutservice","currencyservice","adservice","emailservice","cartservice",
    "productcatalogservice","recommendationservice","paymentservice"
]

intersection_with_possible = sorted([x for x in possible_roots if x in combined_obs_set])

# Top name-field frequencies
def top_name_field(df, name_field, top_n=20):
    if df is None or name_field not in df.columns:
        return pd.DataFrame(columns=[name_field, "count"])
    counts = df[name_field].dropna().value_counts().head(top_n).reset_index()
    counts.columns = [name_field, "count"]
    return counts

metric_top20_kpi = top_name_field(metric, "kpi_name", 20)
trace_top20_trace = top_name_field(trace, "trace_name", 20)
log_top20_log = top_name_field(log, "log_name", 20)

# Compact outputs per file (as requested: tables for top20 cmdb_id + separate lines for intersections and unique lists)
# We'll produce DataFrames/Lists for the controller to display succinctly.

# For readability, also include counts (number of unique cmdb_ids) per file
metric_cmdb_count = len(metric_unique_cmdb_full)
trace_cmdb_count = len(trace_unique_cmdb_full)
log_cmdb_count = len(log_unique_cmdb_full)

# Final displayed variables (compact)
metric_top20_cmdb, metric_cmdb_count, metric_unique_cmdb_limited, metric_top20_kpi, \
trace_top20_cmdb, trace_cmdb_count, trace_unique_cmdb_limited, trace_top20_trace, \
log_top20_cmdb, log_cmdb_count, log_unique_cmdb_limited, log_top20_log, \
intersection_with_possible

```
Out[2]:
```
Summary (compact, plain English):

metric.csv
- Top 20 cmdb_id by row count (cmdb_id — count):
  1. adservice — 20,801
  2. adservice2 — 20,313
  3. frontend2-0 — 11,956
  4. frontend-2 — 11,468
  5. frontend-1 — 11,468
  6. frontend-0 — 11,468
  7. checkoutservice-1 — 8,540
  8. checkoutservice2-0 — 8,540
  9. checkoutservice-2 — 8,296
 10. checkoutservice-0 — 8,296
 11. productcatalogservice-1 — 7,320
 12. productcatalogservice-0 — 7,320
 13. productcatalogservice-2 — 7,076
 14. productcatalogservice2-0 — 6,344
 15. recommendationservice2-0 — 6,222
 16. cartservice-2 — 6,100
 17. cartservice-0 — 6,100
 18. cartservice2-0 — 6,100
 19. recommendationservice-1 — 5,978
 20. recommendationservice-2 — 5,978

- Unique cmdb_id (total 59; first up to 200 shown):
  ['adservice', 'adservice2', 'frontend2-0', 'frontend-2', 'frontend-1', 'frontend-0', 'checkoutservice-1', 'checkoutservice2-0', 'checkoutservice-2', 'checkoutservice-0', 'productcatalogservice-1', 'productcatalogservice-0', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice2-0', 'cartservice-2', 'cartservice-0', 'cartservice2-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice-0', 'adservice2-0', 'cartservice-1', 'adservice-0', 'adservice-2', 'currencyservice-2', 'adservice-1', 'currencyservice2-0', 'shippingservice-0', 'currencyservice-1', 'currencyservice-0', 'shippingservice-2', 'shippingservice-1', 'shippingservice2-0', 'paymentservice-2', 'emailservice2-0', 'emailservice-1', 'emailservice-2', 'emailservice-0', 'paymentservice2-0', 'paymentservice-1', 'paymentservice-0', 'node-6', 'node-2', 'node-5', 'node-3', 'node-1', 'node-4', 'redis-cart2-0', 'redis-cart-0', 'cartservice', 'frontend', 'emailservice', 'checkoutservice', 'currencyservice', 'productcatalogservice', 'paymentservice', 'recommendationservice', 'shippingservice']

- Top 20 kpi_name by frequency (kpi_name — count):
  1. container.node-6.container_network_receive_packets_total — 2,196
  2. container.node-6.container_network_transmit_packets_total — 2,196
  3. container.node-6.container_network_receive_bytes_total_MB — 2,196
  4. container.node-6.container_network_receive_packets — 2,196
  5. container.node-6.container_network_receive_errors_total — 2,196
  6. container.node-6.container_network_transmit_errors_total — 2,196
  7. container.node-6.container_network_transmit_packets — 2,196
  8. container.node-6.container_network_transmit_bytes_total_MB — 2,196
  9. container.node-6.container_memory_swap — 2,074
 10. container.node-6.container_fs_reads./dev/vda1 — 2,074
 11. container.node-6.container_memory_working_set_MB — 2,074
 12. container.node-6.container_fs_inodes./dev/vda1 — 2,074
 13. container.node-6.container_file_descriptors — 2,074
 14. container.node-6.container_memory_max_usage_MB — 2,074
 15. container.node-6.container_memory_rss — 2,074
 16. container.node-6.container_memory_usage_MB — 2,074
 17. container.node-6.container_tasks_state.sleeping — 2,074
 18. container.node-6.container_tasks_state.running — 2,074
 19. container.node-6.container_tasks_state.iowaiting — 2,074
 20. container.node-6.container_fs_sector_reads./dev/vda1 — 2,074

trace.csv
- Top 20 cmdb_id by row count:
  1. frontend-2 — 2,760
  2. frontend-0 — 2,760
  3. checkoutservice-2 — 2,644
  4. checkoutservice-1 — 2,624
  5. checkoutservice-0 — 2,620
  6. frontend-1 — 2,324
  7. productcatalogservice-1 — 1,200
  8. productcatalogservice-2 — 1,200
  9. productcatalogservice-0 — 1,200
 10. frontend2-0 — 1,080
 11. checkoutservice2-0 — 1,080
 12. recommendationservice-0 — 840
 13. recommendationservice-2 — 840
 14. cartservice-0 — 840
 15. cartservice-2 — 840
 16. cartservice-1 — 840
 17. recommendationservice-1 — 840
 18. currencyservice-0 — 720
 19. currencyservice-1 — 720
 20. currencyservice-2 — 720

- Unique cmdb_id (total 40; first up to 200 shown):
  ['frontend-2', 'frontend-0', 'checkoutservice-2', 'checkoutservice-1', 'checkoutservice-0', 'frontend-1', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice-0', 'frontend2-0', 'checkoutservice2-0', 'recommendationservice-0', 'recommendationservice-2', 'cartservice-0', 'cartservice-2', 'cartservice-1', 'recommendationservice-1', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'shippingservice-2', 'shippingservice-0', 'shippingservice-1', 'recommendationservice2-0', 'adservice-0', 'cartservice2-0', 'adservice-2', 'adservice-1', 'paymentservice-1', 'paymentservice-0', 'emailservice-0', 'paymentservice-2', 'emailservice-1', 'emailservice-2', 'shippingservice2-0', 'currencyservice2-0', 'productcatalogservice2-0', 'adservice2-0', 'emailservice2-0', 'paymentservice2-0']

- Top 20 trace_name by frequency (trace_name — count):
  1. trace.from_frontend-0.duration_mean — 630
  2. trace.from_frontend-0.duration_p95 — 630
  3. trace.from_frontend-0.error_rate — 630
  4. trace.from_frontend-0.row_count — 630
  5. trace.from_frontend-2.row_count — 630
  6. trace.from_frontend-2.error_rate — 630
  7. trace.from_frontend-2.duration_p95 — 630
  8. trace.from_frontend-2.duration_mean — 630
  9. trace.from_checkoutservice-2.row_count — 536
 10. trace.from_checkoutservice-2.error_rate — 536
 11. trace.from_checkoutservice-2.duration_p95 — 536
 12. trace.from_checkoutservice-2.duration_mean — 536
 13. trace.from_checkoutservice-1.duration_mean — 531
 14. trace.from_checkoutservice-1.duration_p95 — 531
 15. trace.from_checkoutservice-1.error_rate — 531
 16. trace.from_checkoutservice-1.row_count — 531
 17. trace.from_checkoutservice-0.row_count — 530
 18. trace.from_checkoutservice-0.error_rate — 530
 19. trace.from_checkoutservice-0.duration_p95 — 530
 20. trace.from_checkoutservice-0.duration_mean — 530

log.csv
- Top 20 cmdb_id by row count (only top 20 exist; counts are small):
  1. adservice-2 — 62
  2. cartservice-1 — 62
  3. cartservice-0 — 62
  4. cartservice-2 — 62
  5. currencyservice-2 — 62
  6. currencyservice-0 — 62
  7. currencyservice-1 — 62
  8. recommendationservice-0 — 62
  9. recommendationservice-1 — 62
 10. recommendationservice-2 — 62
 11. productcatalogservice-0 — 62
 12. paymentservice-2 — 62
 13. frontend-2 — 62
 14. frontend-1 — 62
 15. frontend-0 — 62
 16. shippingservice-2 — 62
 17. emailservice-1 — 60
 18. emailservice-0 — 60
 19. checkoutservice-1 — 60
 20. checkoutservice-2 — 60

- Unique cmdb_id (total 31; first up to 200 shown):
  ['adservice-2', 'cartservice-1', 'cartservice-0', 'cartservice-2', 'currencyservice-2', 'currencyservice-0', 'currencyservice-1', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'productcatalogservice-0', 'paymentservice-2', 'frontend-2', 'frontend-1', 'frontend-0', 'shippingservice-2', 'emailservice-1', 'emailservice-0', 'checkoutservice-1', 'checkoutservice-2', 'adservice-0', 'adservice-1', 'checkoutservice-0', 'emailservice-2', 'productcatalogservice-1', 'paymentservice-0', 'paymentservice-1', 'productcatalogservice-2', 'shippingservice-0', 'shippingservice-1', 'redis-cart-0']

- Top log_name by frequency:
  1. log.error_count — 925
  2. log.row_count — 925

Intersection between observed cmdb_ids (union across metric, trace, log) and the provided possible root-cause list:
['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'node-1', 'node-2', 'node-3', 'node-4', 'node-5', 'node-6', 'paymentservice', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'shippingservice', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'shippingservice2-0']

If you want any of these tables exported as CSV-style text, or want counts broken down by a particular cmdb_id, tell me which one and I’ll provide a compact table.

The original code execution output of IPython Kernel is also provided below for reference:

(                       count  count
0                  adservice  20801
1                 adservice2  20313
2                frontend2-0  11956
3                 frontend-2  11468
4                 frontend-1  11468
5                 frontend-0  11468
6          checkoutservice-1   8540
7         checkoutservice2-0   8540
8          checkoutservice-2   8296
9          checkoutservice-0   8296
10   productcatalogservice-1   7320
11   productcatalogservice-0   7320
12   productcatalogservice-2   7076
13  productcatalogservice2-0   6344
14  recommendationservice2-0   6222
15             cartservice-2   6100
16             cartservice-0   6100
17            cartservice2-0   6100
18   recommendationservice-1   5978
19   recommendationservice-2   5978, 59, ['adservice', 'adservice2', 'frontend2-0', 'frontend-2', 'frontend-1', 'frontend-0', 'checkoutservice-1', 'checkoutservice2-0', 'checkoutservice-2', 'checkoutservice-0', 'productcatalogservice-1', 'productcatalogservice-0', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice2-0', 'cartservice-2', 'cartservice-0', 'cartservice2-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice-0', 'adservice2-0', 'cartservice-1', 'adservice-0', 'adservice-2', 'currencyservice-2', 'adservice-1', 'currencyservice2-0', 'shippingservice-0', 'currencyservice-1', 'currencyservice-0', 'shippingservice-2', 'shippingservice-1', 'shippingservice2-0', 'paymentservice-2', 'emailservice2-0', 'emailservice-1', 'emailservice-2', 'emailservice-0', 'paymentservice2-0', 'paymentservice-1', 'paymentservice-0', 'node-6', 'node-2', 'node-5', 'node-3', 'node-1', 'node-4', 'redis-cart2-0', 'redis-cart-0', 'cartservice', 'frontend', 'emailservice', 'checkoutservice', 'currencyservice', 'productcatalogservice', 'paymentservice', 'recommendationservice', 'shippingservice'],                                              kpi_name  count
0   container.node-6.container_network_receive_pac...   2196
1   container.node-6.container_network_transmit_pa...   2196
2   container.node-6.container_network_receive_MB....   2196
3   container.node-6.container_network_receive_pac...   2196
4   container.node-6.container_network_receive_err...   2196
5   container.node-6.container_network_transmit_er...   2196
6   container.node-6.container_network_transmit_pa...   2196
7   container.node-6.container_network_transmit_MB...   2196
8              container.node-6.container_memory_swap   2074
9       container.node-6.container_fs_reads./dev/vda1   2074
10   container.node-6.container_memory_working_set_MB   2074
11     container.node-6.container_fs_inodes./dev/vda1   2074
12        container.node-6.container_file_descriptors   2074
13     container.node-6.container_memory_max_usage_MB   2074
14              container.node-6.container_memory_rss   2074
15         container.node-6.container_memory_usage_MB   2074
16    container.node-6.container_tasks_state.sleeping   2074
17     container.node-6.container_tasks_state.running   2074
18   container.node-6.container_tasks_state.iowaiting   2074
19  container.node-6.container_fs_sector_reads./de...   2074,                       count  count
0                frontend-2   2760
1                frontend-0   2760
2         checkoutservice-2   2644
3         checkoutservice-1   2624
4         checkoutservice-0   2620
5                frontend-1   2324
6   productcatalogservice-1   1200
7   productcatalogservice-2   1200
8   productcatalogservice-0   1200
9               frontend2-0   1080
10       checkoutservice2-0   1080
11  recommendationservice-0    840
12  recommendationservice-2    840
13            cartservice-0    840
14            cartservice-2    840
15            cartservice-1    840
16  recommendationservice-1    840
17        currencyservice-0    720
18        currencyservice-1    720
19        currencyservice-2    720, 40, ['frontend-2', 'frontend-0', 'checkoutservice-2', 'checkoutservice-1', 'checkoutservice-0', 'frontend-1', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice-0', 'frontend2-0', 'checkoutservice2-0', 'recommendationservice-0', 'recommendationservice-2', 'cartservice-0', 'cartservice-2', 'cartservice-1', 'recommendationservice-1', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'shippingservice-2', 'shippingservice-0', 'shippingservice-1', 'recommendationservice2-0', 'adservice-0', 'cartservice2-0', 'adservice-2', 'adservice-1', 'paymentservice-1', 'paymentservice-0', 'emailservice-0', 'paymentservice-2', 'emailservice-1', 'emailservice-2', 'shippingservice2-0', 'currencyservice2-0', 'productcatalogservice2-0', 'adservice2-0', 'emailservice2-0', 'paymentservice2-0'],                                     trace_name  count
0          trace.from_frontend-0.duration_mean    630
1           trace.from_frontend-0.duration_p95    630
2             trace.from_frontend-0.error_rate    630
3              trace.from_frontend-0.row_count    630
4              trace.from_frontend-2.row_count    630
5             trace.from_frontend-2.error_rate    630
6           trace.from_frontend-2.duration_p95    630
7          trace.from_frontend-2.duration_mean    630
8       trace.from_checkoutservice-2.row_count    536
9      trace.from_checkoutservice-2.error_rate    536
10   trace.from_checkoutservice-2.duration_p95    536
11  trace.from_checkoutservice-2.duration_mean    536
12  trace.from_checkoutservice-1.duration_mean    531
13   trace.from_checkoutservice-1.duration_p95    531
14     trace.from_checkoutservice-1.error_rate    531
15      trace.from_checkoutservice-1.row_count    531
16      trace.from_checkoutservice-0.row_count    530
17     trace.from_checkoutservice-0.error_rate    530
18   trace.from_checkoutservice-0.duration_p95    530
19  trace.from_checkoutservice-0.duration_mean    530,                       count  count
0               adservice-2     62
1             cartservice-1     62
2             cartservice-0     62
3             cartservice-2     62
4         currencyservice-2     62
5         currencyservice-0     62
6         currencyservice-1     62
7   recommendationservice-0     62
8   recommendationservice-1     62
9   recommendationservice-2     62
10  productcatalogservice-0     62
11         paymentservice-2     62
12               frontend-2     62
13               frontend-1     62
14               frontend-0     62
15        shippingservice-2     62
16           emailservice-1     60
17           emailservice-0     60
18        checkoutservice-1     60
19        checkoutservice-2     60, 31, ['adservice-2', 'cartservice-1', 'cartservice-0', 'cartservice-2', 'currencyservice-2', 'currencyservice-0', 'currencyservice-1', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'productcatalogservice-0', 'paymentservice-2', 'frontend-2', 'frontend-1', 'frontend-0', 'shippingservice-2', 'emailservice-1', 'emailservice-0', 'checkoutservice-1', 'checkoutservice-2', 'adservice-0', 'adservice-1', 'checkoutservice-0', 'emailservice-2', 'productcatalogservice-1', 'paymentservice-0', 'paymentservice-1', 'productcatalogservice-2', 'shippingservice-0', 'shippingservice-1', 'redis-cart-0'],           log_name  count
0  log.error_count    925
1    log.row_count    925, ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'node-1', 'node-2', 'node-3', 'node-4', 'node-5', 'node-6', 'paymentservice', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'shippingservice', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'shippingservice2-0'])```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing 'metric' and 'intersection_with_possible' if present in the kernel
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

try:
    intersection_with_possible
except NameError:
    # Fallback: derive intersection across files if not present (safe-guard)
    # Collect cmdb_ids from metric, trace, log if present
    sets = []
    if "metric" in globals() and metric is not None and "cmdb_id" in metric.columns:
        sets.append(set(metric["cmdb_id"].dropna().unique()))
    if "trace" in globals() and trace is not None and "cmdb_id" in trace.columns:
        sets.append(set(trace["cmdb_id"].dropna().unique()))
    if "log" in globals() and log is not None and "cmdb_id" in log.columns:
        sets.append(set(log["cmdb_id"].dropna().unique()))
    # use provided possible list from earlier conversation
    possible_roots = [
        "node-1","node-2","node-3","node-4","node-5","node-6",
        "frontend-0","frontend-1","frontend-2","frontend2-0",
        "shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
        "checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
        "currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
        "adservice-0","adservice-1","adservice-2","adservice2-0",
        "emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
        "cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
        "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
        "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
        "paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0",
        "frontend","shippingservice","checkoutservice","currencyservice","adservice","emailservice","cartservice",
        "productcatalogservice","recommendationservice","paymentservice"
    ]
    combined_obs = set().union(*sets) if sets else set()
    intersection_with_possible = sorted([x for x in possible_roots if x in combined_obs])

# Ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 for each (cmdb_id, kpi_name) where cmdb_id in intersection set
mask_intersect = metric['cmdb_id'].isin(intersection_with_possible)
metric_subset_all = metric[mask_intersect].copy()

# Ensure numeric 'value'
metric_subset_all['value'] = pd.to_numeric(metric_subset_all['value'], errors='coerce')

# Compute global P95 per group (using entire available series BEFORE filtering by time)
p95_df = (
    metric_subset_all
    .dropna(subset=['value'])
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95'})
)

# 2) Filter metric.csv to incident window 2022-03-20 14:00:00 to 2022-03-20 14:30:00 Asia/Shanghai (inclusive).
# Convert timestamps to timezone-aware datetimes in Asia/Shanghai
metric_ts = metric.copy()
metric_ts['dt'] = pd.to_datetime(metric_ts['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

start_ts = pd.Timestamp("2022-03-20 14:00:00", tz=tz)
end_ts = pd.Timestamp("2022-03-20 14:30:00", tz=tz)

window_mask = (metric_ts['dt'] >= start_ts) & (metric_ts['dt'] <= end_ts)
# also restrict to intersection cmdb_ids to limit work
window_mask = window_mask & metric_ts['cmdb_id'].isin(intersection_with_possible)
metric_window = metric_ts[window_mask].copy()
metric_window['value'] = pd.to_numeric(metric_window['value'], errors='coerce')

# 3) For each component-KPI compute anomaly_count, earliest anomalous timestamp, max_value in window, p95, severity.

# Compute max_value per group in the window (for severity and reporting)
window_max = (
    metric_window
    .dropna(subset=['value'])
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .max()
    .rename(columns={'value': 'max_value'})
)

# Compute anomaly points by merging window rows with p95 (we need p95 for decision)
window_with_p95 = metric_window.merge(p95_df, on=['cmdb_id','kpi_name'], how='inner')

# Determine anomaly flag per row (respect P95==0 rule)
# If p95 == 0 -> anomaly if value > 0; else anomaly if value >= p95
def is_anom_row(row):
    if pd.isna(row['value']):
        return False
    if pd.isna(row['p95']):
        return False
    if row['p95'] == 0:
        return row['value'] > 0
    else:
        return row['value'] >= row['p95']

# Vectorize
window_with_p95['is_anom'] = np.where(
    window_with_p95['p95'].fillna(0) == 0,
    window_with_p95['value'] > 0,
    window_with_p95['value'] >= window_with_p95['p95']
)

# For anomalous rows, compute earliest timestamp per group
anom_agg = (
    window_with_p95[window_with_p95['is_anom']]
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        anomaly_count=('is_anom', 'sum'),
        earliest_anom_ts=('dt', 'min')
    )
)

# Merge aggregated stats: start from p95_df to ensure global thresholds present
stats = (
    p95_df
    .merge(window_max, on=['cmdb_id','kpi_name'], how='left')
    .merge(anom_agg, on=['cmdb_id','kpi_name'], how='left')
)

# Fill NaNs for groups with no anomalies / no window data
stats['anomaly_count'] = stats['anomaly_count'].fillna(0).astype(int)
# earliest_anom_ts may be NaT
# max_value may be NaN if no points in window

# Compute severity per instructions: if p95==0 -> severity = (max_value - p95) absolute; else (max_value - p95)/p95
# Keep severity as NaN if max_value is NaN (no window data)
def compute_severity(row):
    p95 = row['p95']
    maxv = row['max_value']
    if pd.isna(maxv):
        return np.nan
    if p95 == 0:
        return (maxv - p95)
    else:
        # if p95 != 0
        return (maxv - p95) / p95

stats['severity'] = stats.apply(compute_severity, axis=1)

# Format earliest_anom_ts as string in Asia/Shanghai; keep NaN as NaN
stats['earliest_anom_ts_str'] = stats['earliest_anom_ts'].dt.strftime("%Y-%m-%d %H:%M:%S %Z")

# 4) Return only the top 20 component-KPIs sorted by anomaly_count desc.
out_cols = ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anom_ts_str', 'max_value', 'p95', 'severity']
sorted_by_anom = stats.sort_values(by=['anomaly_count','severity'], ascending=[False, False]).reset_index(drop=True)
top20_by_anom = sorted_by_anom.head(20)[out_cols].rename(columns={'earliest_anom_ts_str':'earliest_anom_ts'})

# 5) If no component-KPI has anomaly_count>0 using P95, return top 20 by largest positive severity instead.
if (top20_by_anom['anomaly_count'].sum() == 0):
    # consider only rows with non-null severity; sort by severity desc
    cand = stats[stats['severity'].notna()].sort_values(by='severity', ascending=False).head(20)
    top20_by_anom = cand[out_cols].rename(columns={'earliest_anom_ts_str':'earliest_anom_ts'})

# Keep final output compact
result = top20_by_anom.reset_index(drop=True).head(20)

result

```
Out[3]:
```
Summary (analysis window: 2022-03-20 14:00:00 to 2022-03-20 14:30:00 Asia/Shanghai; P95 computed using the full metric series per component-KPI):

Top 20 component-KPI records by anomaly count (anomalous = value >= global P95, or >0 when P95==0). All shown groups have anomaly_count = 31 (every minute in the 31-minute window):

1. node-6 — system.disk.total
   - anomaly_count: 31
   - earliest_anom_ts: 2022-03-20 14:00:00 CST
   - max_value: 8.303561e+09
   - p95: 5.101720e+09
   - severity: 0.6276

2. recommendationservice2-0 — container.node-6.container_file_descriptors
   - anomaly_count: 31; earliest: 2022-03-20 14:00:00
   - max: 16.0; p95: 10.0; severity: 0.6

3. checkoutservice-1 — container.node-6.container_sockets
   - anomaly_count: 31; earliest: 2022-03-20 14:00:00
   - max: 4.5; p95: 4.0; severity: 0.125

4. productcatalogservice-1 — container.node-6.container_sockets
   - anomaly_count: 31; earliest: 2022-03-20 14:00:00
   - max: 4.5; p95: 4.0; severity: 0.125

5. node-5 — system.udp.connect.num
   - anomaly_count: 31; earliest: 2022-03-20 14:00:00
   - max: 12.0; p95: 11.0; severity: 0.0909

6. checkoutservice-1 — container.node-6.container_file_descriptors
   - anomaly_count: 31; earliest: 2022-03-20 14:00:00
   - max: 10.5; p95: 10.0; severity: 0.05

7. productcatalogservice-1 — container.node-6.container_file_descriptors
   - anomaly_count: 31; earliest: 2022-03-20 14:00:00
   - max: 10.5; p95: 10.0; severity: 0.05

8. adservice — runtime.jvm_threads_state.RUNNABLE
   - anomaly_count: 31; earliest: 2022-03-20 14:00:00
   - max: 5.083333; p95: 5.0; severity: 0.0167

9. frontend-2 — container.node-5.container_memory_rss
   - anomaly_count: 31; earliest: 2022-03-20 14:00:00
   - max: 1.947238e+07; p95: 1.933722e+07; severity: 0.00699

10. node-3 — system.disk.pct_usage
    - anomaly_count: 31; earliest: 2022-03-20 14:00:00
    - max: 41.85; p95: 41.84; severity: 0.000239

11–19. Multiple adservice KPIs (app.grpc.rr, app.grpc.sr, app.http.count, app.http.rr, app.http.sr, several JVM/runtime metrics)
    - each: anomaly_count = 31, earliest 2022-03-20 14:00:00
    - max equals p95 for many, yielding severity ≈ 0.0

20. adservice — several Java runtime metrics (anomaly_count = 31; severity = 0.0)

Key observations:
- Many component-KPI groups had every point in the incident window flagged as anomalous (anomaly_count = 31), indicating sustained values at or above their global P95 during the entire window.
- The largest relative severities are:
  - node-6 system.disk.total (severity ≈ 0.6276)
  - recommendationservice2-0 container_file_descriptors (severity = 0.6)
  - next highest are much smaller (<= 0.125).
- Several KPIs (notably many adservice metrics) show max == p95 -> severity ~ 0, meaning values reached but did not exceed the global P95.

If you want, I can return this as a compact CSV-like table, or expand to show the full p95 computation counts or include groups with anomaly_count <31 sorted by severity.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count         earliest_anom_ts     max_value           p95  severity
0                     node-6                                  system.disk.total             31  2022-03-20 14:00:00 CST  8.303561e+09  5.101720e+09  0.627600
1   recommendationservice2-0        container.node-6.container_file_descriptors             31  2022-03-20 14:00:00 CST  1.600000e+01  1.000000e+01  0.600000
2          checkoutservice-1                 container.node-6.container_sockets             31  2022-03-20 14:00:00 CST  4.500000e+00  4.000000e+00  0.125000
3    productcatalogservice-1                 container.node-6.container_sockets             31  2022-03-20 14:00:00 CST  4.500000e+00  4.000000e+00  0.125000
4                     node-5                             system.udp.connect.num             31  2022-03-20 14:00:00 CST  1.200000e+01  1.100000e+01  0.090909
5          checkoutservice-1        container.node-6.container_file_descriptors             31  2022-03-20 14:00:00 CST  1.050000e+01  1.000000e+01  0.050000
6    productcatalogservice-1        container.node-6.container_file_descriptors             31  2022-03-20 14:00:00 CST  1.050000e+01  1.000000e+01  0.050000
7                  adservice                 runtime.jvm_threads_state.RUNNABLE             31  2022-03-20 14:00:00 CST  5.083333e+00  5.000000e+00  0.016667
8                 frontend-2              container.node-5.container_memory_rss             31  2022-03-20 14:00:00 CST  1.947238e+07  1.933722e+07  0.006990
9                     node-3                              system.disk.pct_usage             31  2022-03-20 14:00:00 CST  4.185000e+01  4.184000e+01  0.000239
10                 adservice                                        app.grpc.rr             31  2022-03-20 14:00:00 CST  1.000000e+02  1.000000e+02  0.000000
11                 adservice                                        app.grpc.sr             31  2022-03-20 14:00:00 CST  1.000000e+02  1.000000e+02  0.000000
12                 adservice                                     app.http.count             31  2022-03-20 14:00:00 CST  2.400000e+01  2.400000e+01  0.000000
13                 adservice                                        app.http.rr             31  2022-03-20 14:00:00 CST  1.000000e+02  1.000000e+02  0.000000
14                 adservice                                        app.http.sr             31  2022-03-20 14:00:00 CST  1.000000e+02  1.000000e+02  0.000000
15                 adservice    runtime.java_lang_ClassLoading_LoadedClassCount             31  2022-03-20 14:00:00 CST  5.162000e+03  5.162000e+03  0.000000
16                 adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...             31  2022-03-20 14:00:00 CST  5.185000e+03  5.185000e+03  0.000000
17                 adservice  runtime.java_lang_ClassLoading_UnloadedClassCount             31  2022-03-20 14:00:00 CST  2.300000e+01  2.300000e+01  0.000000
18                 adservice  runtime.java_lang_Compilation_CompilationTimeM...             31  2022-03-20 14:00:00 CST  1.000000e+00  1.000000e+00  0.000000
19                 adservice  runtime.java_lang_Compilation_TotalCompilation...             31  2022-03-20 14:00:00 CST  4.701500e+04  4.701500e+04  0.000000```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing 'trace' if present
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric values and timezone-aware datetime in Asia/Shanghai
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace['dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using full series
thr = (
    trace.dropna(subset=['value'])
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
)

# 2) Filter to incident window (inclusive)
start_ts = pd.Timestamp("2022-03-20 14:00:00", tz=tz)
end_ts = pd.Timestamp("2022-03-20 14:30:00", tz=tz)
trace_win = trace[(trace['dt'] >= start_ts) & (trace['dt'] <= end_ts)].copy()

# 3) Define anomaly rule per trace_name (case-insensitive)
# classify: 'high' for duration/error_rate (anomaly when value >= p95, extremal = max)
# 'low' for row_count (anomaly when value <= p05, extremal = min)
def classify(name):
    n = str(name).lower()
    if 'row_count' in n or 'rowcount' in n:
        return 'low'
    if 'error_rate' in n or 'duration' in n:
        return 'high'
    return 'other'

# Attach classification to threshold table and window rows
thr['rule'] = thr['trace_name'].apply(classify)
trace_win = trace_win.merge(thr[['cmdb_id','trace_name','p95','p05','rule']], on=['cmdb_id','trace_name'], how='left')

# 4) Mark anomalies per row according to rule
def mark_anom(row):
    v = row['value']
    if pd.isna(v) or pd.isna(row['rule']):
        return False
    if row['rule'] == 'high':
        if pd.isna(row['p95']):
            return False
        if row['p95'] == 0:
            return v >= row['p95']  # if p95==0 use >= (spec said treat P95==0 specially for metrics; for traces we follow >=)
        return v >= row['p95']
    if row['rule'] == 'low':
        if pd.isna(row['p05']):
            return False
        return v <= row['p05']
    return False

trace_win['is_anom'] = trace_win.apply(mark_anom, axis=1)

# Aggregations per group
# anomaly_count and earliest_anom_ts
anom_agg = (
    trace_win[trace_win['is_anom']]
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(anomaly_count=('is_anom','sum'), earliest_anom_ts=('dt','min'))
)

# extremal_value_in_window: for 'high' -> max(value); for 'low' -> min(value)
extrema_high = (
    trace_win[trace_win['rule']=='high']
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .max()
    .rename(columns={'value':'extremal_value_in_window'})
)
extrema_low = (
    trace_win[trace_win['rule']=='low']
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .min()
    .rename(columns={'value':'extremal_value_in_window'})
)
extrema = pd.concat([extrema_high, extrema_low], ignore_index=True)

# Combine thresholds, extrema, anomalies
groups = thr.merge(extrema, on=['cmdb_id','trace_name'], how='left')
groups = groups.merge(anom_agg, on=['cmdb_id','trace_name'], how='left')

# Fill NaNs for counts
groups['anomaly_count'] = groups['anomaly_count'].fillna(0).astype(int)
# format earliest_anom_ts
groups['earliest_anom_ts_str'] = groups['earliest_anom_ts'].dt.strftime("%Y-%m-%d %H:%M:%S %Z")

# threshold_used and severity calculation
def compute_threshold_and_severity(row):
    rule = row['rule']
    extremal = row['extremal_value_in_window']
    p95 = row['p95']
    p05 = row['p05']
    thr_val = np.nan
    sev = np.nan
    if rule == 'high':
        thr_val = p95
        if pd.isna(extremal):
            sev = np.nan
        else:
            if p95 == 0 or pd.isna(p95):
                sev = abs(extremal - (p95 if not pd.isna(p95) else 0))
            else:
                sev = (extremal - p95) / p95
    elif rule == 'low':
        thr_val = p05
        if pd.isna(extremal):
            sev = np.nan
        else:
            if p05 == 0 or pd.isna(p05):
                sev = abs((p05 if not pd.isna(p05) else 0) - extremal)
            else:
                sev = (p05 - extremal) / p05
    return pd.Series({'threshold_used': thr_val, 'severity': sev})

thr_sev = groups.apply(compute_threshold_and_severity, axis=1)
groups = pd.concat([groups, thr_sev], axis=1)

# Prepare final columns
groups['earliest_anom_ts'] = groups['earliest_anom_ts_str']
out = groups[['cmdb_id','trace_name','anomaly_count','earliest_anom_ts','extremal_value_in_window','threshold_used','severity']].copy()

# 5) Return top 20 groups sorted by anomaly_count desc (include ties)
out_sorted = out.sort_values(by=['anomaly_count','severity'], ascending=[False, False]).reset_index(drop=True)

# include ties at the 20th position
if len(out_sorted) <= 20:
    top_groups = out_sorted
else:
    cutoff_count = out_sorted.iloc[19]['anomaly_count']
    top_groups = out_sorted[out_sorted['anomaly_count'] >= cutoff_count].reset_index(drop=True)

# Count distinct groups with anomaly_count > 0
num_groups_with_anom = int((out['anomaly_count'] > 0).sum())

# Keep output compact: top_groups (table) and a one-line summary string
summary_line = f"{num_groups_with_anom} distinct cmdb_id/trace_name groups had anomaly_count>0 in the window."

top_groups.head(200), summary_line

```
Out[4]:
```
# Summarize the trace anomaly analysis in plain English as a single concise string.
try:
    summary_line  # from previous execution
except NameError:
    summary_line = None

try:
    top_groups  # from previous execution
except NameError:
    top_groups = None

# Build a compact plain-English summary based on available variables
if summary_line is None or top_groups is None:
    summary_msg = "Trace analysis results are not available in the current session."
else:
    # parse total groups with anomalies from summary_line if possible
    try:
        total_groups = int(summary_line.split()[0])
    except Exception:
        total_groups = summary_line

    # capture top 3 example groups for quick illustration
    examples = []
    for _, r in top_groups.head(3).iterrows():
        cid = r['cmdb_id']
        tname = r['trace_name']
        cnt = int(r['anomaly_count']) if not pd.isna(r['anomaly_count']) else None
        ts = r['earliest_anom_ts'] if pd.notna(r.get('earliest_anom_ts')) else "N/A"
        extremal = r['extremal_value_in_window']
        thr = r['threshold_used']
        sev = r['severity']
        examples.append(f"{cid} | {tname} | anomalies={cnt} | earliest={ts} | extremal={extremal} | threshold={thr} | severity={sev}")

    # Compose summary
    summary_msg = (
        f"Trace-level anomaly summary (window 2022-03-20 14:00–14:30 Asia/Shanghai): "
        f"{total_groups} distinct cmdb_id/trace_name groups had anomaly_count>0.\n"
        "Most anomalous groups are duration/error_rate traces flagged when value >= global P95; "
        "row_count traces would be flagged for low values <= P05. Many groups show sustained anomalies "
        "throughout the window (anomaly_count typically 30), with earliest anomalies at the window start.\n"
        "Top example groups (3 samples):\n- " + "\n- ".join(examples) + "\n"
        "Note: a large fraction of thresholds (P95) are zero, so non-zero trace values were counted as anomalies, "
        "leading to many groups with small absolute severities. Recommend prioritizing groups with largest positive "
        "severity for investigation rather than raw anomaly count."
    )

summary_msg

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                  trace_name  anomaly_count         earliest_anom_ts  extremal_value_in_window  threshold_used  severity
0        cartservice-1                     trace.self.duration_p95             30  2022-03-20 14:00:00 CST                    0.0110           0.000    0.0110
1        cartservice-0   trace.from_checkoutservice-0.duration_p95             30  2022-03-20 14:00:00 CST                    0.0019           0.000    0.0019
2    checkoutservice-0         trace.to_cartservice-0.duration_p95             30  2022-03-20 14:00:00 CST                    0.0019           0.000    0.0019
3        cartservice-0  trace.from_checkoutservice-0.duration_mean             30  2022-03-20 14:00:00 CST                    0.0010           0.000    0.0010
4        cartservice-2                     trace.self.duration_p95             30  2022-03-20 14:00:00 CST                    0.0010           0.000    0.0010
..                 ...                                         ...            ...                      ...                       ...             ...       ...
195         frontend-2       trace.to_shippingservice-2.error_rate             30  2022-03-20 14:00:00 CST                    0.0000           0.000    0.0000
196        frontend2-0                  trace.from_root.error_rate             30  2022-03-20 14:00:00 CST                    0.0000           0.000    0.0000
197        frontend2-0                       trace.self.error_rate             30  2022-03-20 14:00:00 CST                    0.0000           0.000    0.0000
198        frontend2-0            trace.to_adservice2-0.error_rate             30  2022-03-20 14:00:00 CST                    0.0000           0.000    0.0000
199        frontend2-0        trace.to_cartservice2-0.duration_p95             30  2022-03-20 14:00:00 CST                    0.0010           0.001    0.0000

[200 rows x 7 columns], '1232 distinct cmdb_id/trace_name groups had anomaly_count>0 in the window.')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse 'metric' and 'tz' if already in the session; otherwise load/define
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric value and datetime in Asia/Shanghai
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
if 'dt' not in metric.columns:
    metric['dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter rows where kpi_name contains 'container.node-6'
mask_cn6 = metric['kpi_name'].astype(str).str.contains('container.node-6', na=False)
metric_cn6_all = metric[mask_cn6].copy()

# Distinct cmdb_id list (limit to 200)
cn6_cmdb_list_full = sorted(metric_cn6_all['cmdb_id'].dropna().unique().tolist())
cn6_cmdb_list_limited = cn6_cmdb_list_full[:200]

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series
p95_df = (
    metric_cn6_all
    .dropna(subset=['value'])
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value':'p95'})
)

# 2) Filter to incident window inclusive
start_ts = pd.Timestamp("2022-03-20 14:00:00", tz=tz)
end_ts = pd.Timestamp("2022-03-20 14:30:00", tz=tz)
metric_cn6_win = metric_cn6_all[(metric_cn6_all['dt'] >= start_ts) & (metric_cn6_all['dt'] <= end_ts)].copy()

# Merge p95 into window rows
win_with_p95 = metric_cn6_win.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')

# Determine anomaly per row: if p95==0 -> anomalous when value>0; else value>=p95
win_with_p95['is_anom'] = np.where(
    win_with_p95['p95'].fillna(0) == 0,
    win_with_p95['value'] > 0,
    win_with_p95['value'] >= win_with_p95['p95']
)

# 2) Count anomalous timestamps per (cmdb_id,kpi_name) and earliest anomaly time
group_anom_counts = (
    win_with_p95[win_with_p95['is_anom']]
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(anom_count=('is_anom','sum'), earliest_anom_ts=('dt','min'))
)

# Compute max value in window per group (for severity)
group_max_in_window = (
    metric_cn6_win.dropna(subset=['value'])
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .max()
    .rename(columns={'value':'max_in_window'})
)

# Combine p95, max_in_window, and anom_counts
group_stats = p95_df.merge(group_max_in_window, on=['cmdb_id','kpi_name'], how='left') \
                    .merge(group_anom_counts, on=['cmdb_id','kpi_name'], how='left')

group_stats['anom_count'] = group_stats['anom_count'].fillna(0).astype(int)

# Compute severity per group
def compute_sev(row):
    p95 = row['p95']
    maxv = row['max_in_window']
    if pd.isna(maxv):
        return np.nan
    if pd.isna(p95) or p95 == 0:
        base = 0 if pd.isna(p95) else p95
        return (maxv - base)
    else:
        return (maxv - p95) / p95

group_stats['severity'] = group_stats.apply(compute_sev, axis=1)

# 3) Aggregate per cmdb_id:
# anomalous_kpi_count = number of distinct kpi_names with anom_count > 0
anom_kpi_counts = (
    group_stats[group_stats['anom_count'] > 0]
    .groupby('cmdb_id', as_index=False)
    .agg(
        anomalous_kpi_count=('kpi_name','nunique'),
        earliest_anom_ts=('earliest_anom_ts','min'),
        max_severity=('severity','max')
    )
)

# Ensure all cmdb_ids appear (with zeros for those without anomalies)
all_cmdb_df = pd.DataFrame({'cmdb_id': cn6_cmdb_list_limited})
anom_kpi_counts_full = all_cmdb_df.merge(anom_kpi_counts, on='cmdb_id', how='left')
anom_kpi_counts_full['anomalous_kpi_count'] = anom_kpi_counts_full['anomalous_kpi_count'].fillna(0).astype(int)
# earliest_anom_ts may be NaT; handle formatting robustly
def format_dt_series(s):
    # s is a Series possibly tz-aware or naive
    try:
        tzinfo = s.dt.tz
    except Exception:
        tzinfo = None
    if tzinfo is not None:
        out = s.dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %Z")
    else:
        out = pd.to_datetime(s, errors='coerce', utc=True).dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %Z")
    # replace 'NaT' strings with pd.NA
    out = out.replace('NaT', pd.NA)
    return out

# apply formatting on the merged earliest_anom_ts column (which may be object if missing)
if 'earliest_anom_ts' in anom_kpi_counts_full.columns:
    anom_kpi_counts_full['earliest_anom_ts'] = format_dt_series(anom_kpi_counts_full['earliest_anom_ts'])
else:
    anom_kpi_counts_full['earliest_anom_ts'] = pd.NA

anom_kpi_counts_full['max_severity'] = anom_kpi_counts_full['max_severity'].astype(float)

# 4) Prepare final compact table: top 20 cmdb_id sorted by anomalous_kpi_count desc
result = anom_kpi_counts_full.sort_values(by=['anomalous_kpi_count','max_severity'], ascending=[False, False]) \
    .reset_index(drop=True)[['cmdb_id','anomalous_kpi_count','earliest_anom_ts','max_severity']].head(20)

# Return compact outputs
result, cn6_cmdb_list_limited

```
Out[5]:
```
summary = (
    "Summary: container.node-6 KPIs (window 2022-03-20 14:00–14:30 Asia/Shanghai)\n\n"
    "Key points:\n"
    "- 36 distinct cmdb_id values had at least one kpi_name containing 'container.node-6'.\n"
    "- Earliest anomalous timestamps for reported groups are at 2022-03-20 14:00:00 CST (start of window).\n\n"
    "Top cmdb_ids by number of anomalous container.node-6 KPIs (anomalous_kpi_count, max_severity):\n"
    "1) shippingservice-0 — anomalous_kpi_count=38, max_severity=309.0\n"
    "2) shippingservice2-0 — anomalous_kpi_count=38, max_severity=33.0\n"
    "3) cartservice-0 — anomalous_kpi_count=35, max_severity=111.0\n"
    "4) cartservice-2 — anomalous_kpi_count=35, max_severity=33.0\n"
    "5) shippingservice-1 — anomalous_kpi_count=35, max_severity=19.0\n"
    "6) recommendationservice2-0 — anomalous_kpi_count=34, max_severity=14652.0 (very large)\n"
    "7) cartservice-1 — anomalous_kpi_count=33, max_severity=1010.5\n"
    "8) paymentservice-0 — anomalous_kpi_count=32, max_severity=20.461172\n"
    "9) frontend-0 — anomalous_kpi_count=31, max_severity=33.0\n"
    "10) productcatalogservice-2 — anomalous_kpi_count=31, max_severity=16.5\n\n"
    "Interpretation:\n"
    "- Many components show multiple container.node-6 KPIs exceeding their global P95 for the entire window (counts are high and earliest occurrences at the window start).\n"
    "- recommendationservice2-0, cartservice-1 and shippingservice-0 stand out by very large max_severity values and should be prioritized for investigation.\n\n"
    "Full list of observed cmdb_id values with 'container.node-6' KPIs (up to 200):\n"
    "['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2',\n"
    " 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1',\n"
    " 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2',\n"
    " 'emailservice2-0', 'frontend-0', 'frontend2-0', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2',\n"
    " 'paymentservice2-0', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2',\n"
    " 'productcatalogservice2-0', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2',\n"
    " 'recommendationservice2-0', 'redis-cart-0', 'redis-cart2-0', 'shippingservice-0', 'shippingservice-1',\n"
    " 'shippingservice2-0']"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  anomalous_kpi_count         earliest_anom_ts  max_severity
0          shippingservice-0                   38  2022-03-20 14:00:00 CST    309.000000
1         shippingservice2-0                   38  2022-03-20 14:00:00 CST     33.000000
2              cartservice-0                   35  2022-03-20 14:00:00 CST    111.000000
3              cartservice-2                   35  2022-03-20 14:00:00 CST     33.000000
4          shippingservice-1                   35  2022-03-20 14:00:00 CST     19.000000
5   recommendationservice2-0                   34  2022-03-20 14:00:00 CST  14652.000000
6              cartservice-1                   33  2022-03-20 14:00:00 CST   1010.500000
7           paymentservice-0                   32  2022-03-20 14:00:00 CST     20.461172
8                 frontend-0                   31  2022-03-20 14:00:00 CST     33.000000
9    productcatalogservice-2                   31  2022-03-20 14:00:00 CST     16.500000
10         currencyservice-1                   31  2022-03-20 14:00:00 CST      6.575644
11        currencyservice2-0                   31  2022-03-20 14:00:00 CST      1.552199
12          paymentservice-2                   30  2022-03-20 14:00:00 CST     20.592603
13         currencyservice-0                   30  2022-03-20 14:00:00 CST      6.163959
14         currencyservice-2                   30  2022-03-20 14:00:00 CST      1.000000
15         checkoutservice-1                   29  2022-03-20 14:00:00 CST      7.931441
16   productcatalogservice-1                   29  2022-03-20 14:00:00 CST      1.000000
17         checkoutservice-0                   29  2022-03-20 14:00:00 CST      0.412099
18  productcatalogservice2-0                   29  2022-03-20 14:00:00 CST      0.283160
19               adservice-1                   29  2022-03-20 14:00:00 CST      0.200000, ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend-0', 'frontend2-0', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'redis-cart-0', 'redis-cart2-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice2-0'])```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing 'metric' if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# Ensure numeric and timezone-aware datetime in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
metric['dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series BEFORE filtering
p95_all = (
    metric.dropna(subset=['value'])
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value':'p95'})
)

# 2) Filter to the incident window (inclusive)
start_ts = pd.Timestamp("2022-03-20 14:00:00", tz=tz)
end_ts = pd.Timestamp("2022-03-20 14:30:00", tz=tz)
metric_win = metric[(metric['dt'] >= start_ts) & (metric['dt'] <= end_ts)].copy()

# Merge p95 into window rows
win = metric_win.merge(p95_all, on=['cmdb_id','kpi_name'], how='left')

# 3) Determine anomaly per row: if p95==0 -> anomalous when value>0; else value>=p95
win['is_anom'] = np.where(
    win['p95'].fillna(0) == 0,
    win['value'] > 0,
    win['value'] >= win['p95']
)

# 4) Aggregate per group: anomaly_count, earliest_anom_ts, max_value (in-window), p95 already present
anom_groups = (
    win[win['is_anom']]
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        anomaly_count=('is_anom','sum'),
        earliest_anom_ts=('dt','min')
    )
)

max_in_window = (
    win.dropna(subset=['value'])
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .max()
    .rename(columns={'value':'max_value'})
)

# Combine stats
group_stats = p95_all.merge(max_in_window, on=['cmdb_id','kpi_name'], how='left') \
                    .merge(anom_groups, on=['cmdb_id','kpi_name'], how='left')

group_stats['anomaly_count'] = group_stats['anomaly_count'].fillna(0).astype(int)
# earliest_anom_ts may be NaT; keep as datetime
# 5) Compute severity per instructions
def compute_severity(row):
    p95 = row['p95']
    maxv = row['max_value']
    if pd.isna(maxv):
        return np.nan
    if pd.isna(p95) or p95 == 0:
        base = 0 if pd.isna(p95) else p95
        return (maxv - base)
    else:
        return (maxv - p95) / p95

group_stats['severity'] = group_stats.apply(compute_severity, axis=1)

# 6) Select groups whose earliest_anom_ts is strictly greater than window start
# earliest_anom_ts exists only for groups with anomaly_count>0; filter those first
groups_with_anom = group_stats[group_stats['anomaly_count'] > 0].copy()
# keep only if earliest_anom_ts > start_ts
groups_with_anom = groups_with_anom[pd.to_datetime(groups_with_anom['earliest_anom_ts']) > start_ts]

# Format earliest_anom_ts as string in Asia/Shanghai for compact display
groups_with_anom['earliest_anom_ts'] = pd.to_datetime(groups_with_anom['earliest_anom_ts']).dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %Z")

# Keep required columns and sort by earliest_anom_ts ascending, then limit to top 20
result_top20 = groups_with_anom[['cmdb_id','kpi_name','earliest_anom_ts','anomaly_count','max_value','p95','severity']] \
    .sort_values(by=['earliest_anom_ts','severity'], ascending=[True, False]) \
    .head(20) \
    .reset_index(drop=True)

# Total count of distinct groups that started after 14:00:00
count_after = int(groups_with_anom.shape[0])

# Compact outputs
result_top20, count_after

```
Out[6]:
```
Summary (using Asia/Shanghai time; incident window 2022-03-20 14:00–14:30):

- Overall: 1,383 distinct component–KPI groups had their earliest anomalous point strictly after 2022-03-20 14:00:00 (i.e., anomalies that began later than the window start).

- Pattern: Most of the top groups show earliest_anom_ts = 2022-03-20 14:01:00 CST (one minute after window start), indicating anomalies often began immediately after the window began rather than exactly at 14:00.

- Top example groups (sorted by earliest_anom_ts then severity; all started after 14:00):
  1. node-5 — system.io.avg_q_sz
     - earliest_anom_ts: 2022-03-20 14:01:00 CST
     - anomaly_count: 3
     - max_value (in-window): 0.04
     - global P95: 0.02
     - severity: 1.00 (100% relative increase)
  2. node-2 — system.net.tcp.out_segs
     - earliest_anom_ts: 2022-03-20 14:01:00 CST
     - anomaly_count: 2; max_value: 4825.9; p95: 2861.25; severity ≈ 0.687
  3. node-6 — system.net.tcp.in_segs
     - earliest_anom_ts: 2022-03-20 14:01:00 CST
     - anomaly_count: 4; max_value: 358.57; p95: 233.97; severity ≈ 0.533
  4. node-6 — system.net.tcp.retrans_segs
     - earliest_anom_ts: 2022-03-20 14:01:00 CST
     - anomaly_count: 5; max_value: 0.03; p95: 0.02; severity = 0.5
  5. node-5 — system.io.util
     - earliest_anom_ts: 2022-03-20 14:01:00 CST
     - anomaly_count: 4; max_value: 1.3; p95: 0.9; severity ≈ 0.444

- Additional notes:
  - The returned top-20 rows (examples above) include both node-level system metrics (IO, network, file-descriptor/NOFILE usage) and some JVM/runtime metrics for services (e.g., adservice2 CPU/GarbageCollector metrics).
  - Severity is computed as (max_in_window − P95)/P95 (or absolute difference when P95==0). Several groups show large relative increases; many groups have modest severities but still nonzero anomaly counts.
  - Because many anomalies have earliest timestamp 14:01:00, investigation should inspect events occurring around 14:00–14:02 to identify triggers or cascades.

If you want, I can return the full table of groups that started after 14:00 (up to a specified limit), or filter to groups with severity above a threshold for focused investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name         earliest_anom_ts  anomaly_count     max_value            p95  severity
0       node-5                                 system.io.avg_q_sz  2022-03-20 14:01:00 CST              3  4.000000e-02       0.020000  1.000000
1       node-2                            system.net.tcp.out_segs  2022-03-20 14:01:00 CST              2  4.825900e+03    2861.250000  0.686640
2       node-6                             system.net.tcp.in_segs  2022-03-20 14:01:00 CST              4  3.585700e+02     233.970000  0.532547
3       node-6                        system.net.tcp.retrans_segs  2022-03-20 14:01:00 CST              5  3.000000e-02       0.020000  0.500000
4       node-5                                     system.io.util  2022-03-20 14:01:00 CST              4  1.300000e+00       0.900000  0.444444
5       node-2                              system.net.bytes_sent  2022-03-20 14:01:00 CST              2  1.213134e+06  919728.070000  0.319014
6       node-5                                      system.io.w_s  2022-03-20 14:01:00 CST              4  4.500000e+01      38.000000  0.184211
7       node-2                             system.net.tcp.in_segs  2022-03-20 14:01:00 CST              2  1.316950e+03    1200.680000  0.096837
8       node-2                        system.net.packets_in.count  2022-03-20 14:01:00 CST              1  5.509700e+02     502.920000  0.095542
9       node-2                              system.net.bytes_rcvd  2022-03-20 14:01:00 CST              1  6.897076e+05  631489.000000  0.092193
10      node-2                       system.net.udp.out_datagrams  2022-03-20 14:01:00 CST              2  1.270000e+00       1.170000  0.085470
11      node-3                                      system.io.w_s  2022-03-20 14:01:00 CST              3  5.350000e+01      49.500000  0.080808
12  adservice2  runtime.java_lang_GarbageCollector_LastGcInfo_...  2022-03-20 14:01:00 CST              4  2.250000e+00       2.083333  0.080000
13  adservice2   runtime.java_lang_OperatingSystem_ProcessCpuLoad  2022-03-20 14:01:00 CST              4  2.218012e-04       0.000211  0.049328
14      node-4                            system.net.tcp.out_segs  2022-03-20 14:01:00 CST              3  2.079400e+03    2012.030000  0.033484
15      node-4                             system.net.tcp.in_segs  2022-03-20 14:01:00 CST              4  5.828200e+02     570.880000  0.020915
16      node-4                              system.net.bytes_sent  2022-03-20 14:01:00 CST              3  5.242192e+05  517078.000000  0.013811
17      node-1                           system.os.nofile.current  2022-03-20 14:01:00 CST              5  2.688000e+03    2656.000000  0.012048
18      node-3                             system.net.tcp.in_segs  2022-03-20 14:01:00 CST              3  7.852500e+02     778.560000  0.008593
19      node-3                          system.os.nofile.used_pct  2022-03-20 14:01:00 CST             15  2.430000e-01       0.242000  0.004132, 1383)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables if present; otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# Try to reuse previously computed cmdb list from container.node-6 analysis
try:
    cn6_cmdb_list_full
except NameError:
    try:
        cn6_cmdb_list_limited
    except NameError:
        # fallback: derive from metric.csv now
        import os
        if os.path.exists("metric.csv"):
            m = pd.read_csv("metric.csv")
            cn6_cmdb_list_full = sorted(m[m['kpi_name'].astype(str).str.contains('container.node-6', na=False)]['cmdb_id'].dropna().unique().tolist())
        else:
            cn6_cmdb_list_full = []
    else:
        cn6_cmdb_list_full = cn6_cmdb_list_limited

# Ensure timezone and numeric
tz = pytz.timezone('Asia/Shanghai')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace['dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Restrict trace to cmdb_ids of interest
trace_sub = trace[trace['cmdb_id'].isin(cn6_cmdb_list_full)].copy()

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using full series (no time filter)
thr = (
    trace_sub.dropna(subset=['value'])
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
)

# 2) Filter to incident window (inclusive)
start_ts = pd.Timestamp("2022-03-20 14:00:00", tz=tz)
end_ts = pd.Timestamp("2022-03-20 14:30:00", tz=tz)
trace_win = trace_sub[(trace_sub['dt'] >= start_ts) & (trace_sub['dt'] <= end_ts)].copy()

# Merge thresholds into window rows
trace_win = trace_win.merge(thr, on=['cmdb_id','trace_name'], how='left')

# 3) Classify trace_name: 'high' for duration/error_rate, 'low' for row_count
def classify(name):
    n = str(name).lower()
    if 'row_count' in n or 'rowcount' in n:
        return 'low'
    if 'error_rate' in n or 'duration' in n:
        return 'high'
    return 'other'

trace_win['rule'] = trace_win['trace_name'].apply(classify)

# 4) Mark anomalies per rules:
# high: value >= p95
# low: value <= p05
def is_anom(r):
    v = r['value']
    if pd.isna(v):
        return False
    if r['rule'] == 'high':
        if pd.isna(r['p95']):
            return False
        return v >= r['p95']
    if r['rule'] == 'low':
        if pd.isna(r['p05']):
            return False
        return v <= r['p05']
    return False

trace_win['is_anom'] = trace_win.apply(is_anom, axis=1)

# Compute extremal_value_in_window per group depending on rule
extrema_high = (
    trace_win[trace_win['rule'] == 'high']
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .max()
    .rename(columns={'value':'extremal_value_in_window'})
)
extrema_low = (
    trace_win[trace_win['rule'] == 'low']
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .min()
    .rename(columns={'value':'extremal_value_in_window'})
)
extrema = pd.concat([extrema_high, extrema_low], ignore_index=True)

# anomaly_count and earliest_anom_ts
anom_agg = (
    trace_win[trace_win['is_anom']]
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(anomaly_count=('is_anom','sum'), earliest_anom_ts=('dt','min'))
)

# Combine thresholds, extrema, and anomaly aggregates
groups = thr.merge(extrema, on=['cmdb_id','trace_name'], how='left')
groups = groups.merge(anom_agg, on=['cmdb_id','trace_name'], how='left')

groups['anomaly_count'] = groups['anomaly_count'].fillna(0).astype(int)
groups['earliest_anom_ts'] = groups['earliest_anom_ts'].dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %Z")

# Determine threshold_used and severity per group
def thresh_and_sev(row):
    rule = classify(row['trace_name'])
    extremal = row.get('extremal_value_in_window', np.nan)
    p95 = row.get('p95', np.nan)
    p05 = row.get('p05', np.nan)
    thr_val = np.nan
    sev = np.nan
    if rule == 'high':
        thr_val = p95
        if pd.isna(extremal):
            sev = np.nan
        else:
            if pd.isna(p95) or p95 == 0:
                base = 0 if pd.isna(p95) else p95
                sev = abs(extremal - base)
            else:
                sev = (extremal - p95) / p95
    elif rule == 'low':
        thr_val = p05
        if pd.isna(extremal):
            sev = np.nan
        else:
            if pd.isna(p05) or p05 == 0:
                base = 0 if pd.isna(p05) else p05
                sev = abs(base - extremal)
            else:
                sev = (p05 - extremal) / p05
    return pd.Series({'threshold_used': thr_val, 'severity': sev})

thr_sev = groups.apply(thresh_and_sev, axis=1)
groups = pd.concat([groups, thr_sev], axis=1)

# Prepare output table with required columns
out = groups[['cmdb_id','trace_name','anomaly_count','earliest_anom_ts','extremal_value_in_window','threshold_used','severity']].copy()

# 5) Top 20 rows sorted by anomaly_count desc (include ties)
out_sorted = out.sort_values(by=['anomaly_count','severity'], ascending=[False, False]).reset_index(drop=True)
if len(out_sorted) <= 20:
    top_groups = out_sorted
else:
    cutoff = out_sorted.iloc[19]['anomaly_count']
    top_groups = out_sorted[out_sorted['anomaly_count'] >= cutoff].reset_index(drop=True)

# Count distinct groups with anomaly_count > 0
num_groups_with_anom = int((out['anomaly_count'] > 0).sum())

# Compact display variables
top_groups.head(20), num_groups_with_anom

```
Out[7]:
```
Summary (trace analysis restricted to cmdb_ids that had "container.node-6" KPIs; window 2022-03-20 14:00–14:30 Asia/Shanghai):

- Total groups with anomalies: 908 distinct cmdb_id / trace_name groups had anomaly_count > 0 in the window.

- Pattern:
  - Many groups show sustained anomalies throughout the window (anomaly_count typically = 30) and have earliest_anom_ts = 2022-03-20 14:00:00 CST.
  - A large fraction of trace thresholds (P95 or P05) are zero, so any non-zero trace values were flagged as anomalous; this produces many groups with nonzero anomaly counts but small absolute severities.

- Top example groups (top by anomaly_count; anomaly_count, earliest_anom_ts, extremal_value_in_window, threshold_used, severity):
  1. cartservice-1 | trace.self.duration_p95 — anomalies=30, earliest=2022-03-20 14:00:00 CST, extremal=0.011, threshold=0.0, severity=0.011
  2. cartservice-0 | trace.from_checkoutservice-0.duration_p95 — anomalies=30, earliest=14:00, extremal=0.0019, threshold=0.0, severity=0.0019
  3. checkoutservice-0 | trace.to_cartservice-0.duration_p95 — anomalies=30, earliest=14:00, extremal=0.0019, threshold=0.0, severity=0.0019
  4. cartservice-0 | trace.from_checkoutservice-0.duration_mean — anomalies=30, earliest=14:00, extremal=0.0010, threshold=0.0, severity=0.0010
  5. cartservice-2 | trace.self.duration_p95 — anomalies=30, earliest=14:00, extremal=0.0010, threshold=0.0, severity=0.0010
  - (Also many adservice.* error_rate traces appear with anomalies=30 but extremal=0.0 and severity=0.0.)

- Interpretation / next steps:
  - Because many thresholds are zero, anomaly_count alone is noisy. Prioritize groups with meaningful positive severity (largest extremal − threshold relative to threshold) rather than raw counts.
  - Many anomalous traces are duration metrics between cart/checkout services and error_rate traces for adservice; investigate service interactions around 14:00 and nodes with container.node-6 KPIs for root causes.

If you want, I can: (a) show the top groups sorted by severity instead of count, (b) return the full top-200 anomalous groups, or (c) produce a short prioritized list of groups with severity above a chosen threshold.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                  trace_name  anomaly_count         earliest_anom_ts  extremal_value_in_window  threshold_used  severity
0       cartservice-1                     trace.self.duration_p95             30  2022-03-20 14:00:00 CST                  0.011000             0.0  0.011000
1       cartservice-0   trace.from_checkoutservice-0.duration_p95             30  2022-03-20 14:00:00 CST                  0.001900             0.0  0.001900
2   checkoutservice-0         trace.to_cartservice-0.duration_p95             30  2022-03-20 14:00:00 CST                  0.001900             0.0  0.001900
3       cartservice-0  trace.from_checkoutservice-0.duration_mean             30  2022-03-20 14:00:00 CST                  0.001000             0.0  0.001000
4       cartservice-2                     trace.self.duration_p95             30  2022-03-20 14:00:00 CST                  0.001000             0.0  0.001000
5   checkoutservice-0        trace.to_cartservice-0.duration_mean             30  2022-03-20 14:00:00 CST                  0.001000             0.0  0.001000
6       cartservice-1   trace.from_checkoutservice-1.duration_p95             30  2022-03-20 14:00:00 CST                  0.000950             0.0  0.000950
7   checkoutservice-1         trace.to_cartservice-1.duration_p95             30  2022-03-20 14:00:00 CST                  0.000950             0.0  0.000950
8       cartservice-2   trace.from_checkoutservice-1.duration_p95             30  2022-03-20 14:00:00 CST                  0.000900             0.0  0.000900
9   checkoutservice-1         trace.to_cartservice-2.duration_p95             30  2022-03-20 14:00:00 CST                  0.000900             0.0  0.000900
10      cartservice-1  trace.from_checkoutservice-1.duration_mean             30  2022-03-20 14:00:00 CST                  0.000500             0.0  0.000500
11  checkoutservice-1        trace.to_cartservice-1.duration_mean             30  2022-03-20 14:00:00 CST                  0.000500             0.0  0.000500
12      cartservice-2  trace.from_checkoutservice-1.duration_mean             30  2022-03-20 14:00:00 CST                  0.000333             0.0  0.000333
13  checkoutservice-1        trace.to_cartservice-2.duration_mean             30  2022-03-20 14:00:00 CST                  0.000333             0.0  0.000333
14        adservice-0            trace.from_frontend-0.error_rate             30  2022-03-20 14:00:00 CST                  0.000000             0.0  0.000000
15        adservice-0            trace.from_frontend-1.error_rate             30  2022-03-20 14:00:00 CST                  0.000000             0.0  0.000000
16        adservice-0            trace.from_frontend-2.error_rate             30  2022-03-20 14:00:00 CST                  0.000000             0.0  0.000000
17        adservice-1            trace.from_frontend-0.error_rate             30  2022-03-20 14:00:00 CST                  0.000000             0.0  0.000000
18        adservice-1            trace.from_frontend-1.error_rate             30  2022-03-20 14:00:00 CST                  0.000000             0.0  0.000000
19        adservice-1            trace.from_frontend-2.error_rate             30  2022-03-20 14:00:00 CST                  0.000000             0.0  0.000000, 908)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse trace and cn6_cmdb_list_full if available; otherwise load/derive
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

try:
    cn6_cmdb_list_full
except NameError:
    # derive from metric if needed
    try:
        metric
    except NameError:
        metric = pd.read_csv("metric.csv")
    cn6_cmdb_list_full = sorted(metric[metric['kpi_name'].astype(str).str.contains('container.node-6', na=False)]['cmdb_id'].dropna().unique().tolist())

tz = pytz.timezone('Asia/Shanghai')

# Ensure types and timezone-aware datetimes
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace['dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Restrict to cmdb_ids of interest and trace_names containing 'error_rate' or 'row_count' (case-insensitive)
mask_cmdb = trace['cmdb_id'].isin(cn6_cmdb_list_full)
mask_name = trace['trace_name'].astype(str).str.lower().str.contains('error_rate') | trace['trace_name'].astype(str).str.lower().str.contains('row_count')
trace_focus = trace[mask_cmdb & mask_name].copy()

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using full series BEFORE time filtering
thr = (
    trace_focus.dropna(subset=['value'])
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
)

# 2) Filter to incident window (inclusive)
start_ts = pd.Timestamp("2022-03-20 14:00:00", tz=tz)
end_ts = pd.Timestamp("2022-03-20 14:30:00", tz=tz)
trace_win = trace_focus[(trace_focus['dt'] >= start_ts) & (trace_focus['dt'] <= end_ts)].copy()

# Merge thresholds
trace_win = trace_win.merge(thr, on=['cmdb_id','trace_name'], how='left')

# 3) Classify rule: 'high' for error_rate (we only selected error_rate and row_count)
trace_win['rule'] = trace_win['trace_name'].astype(str).str.lower().apply(lambda s: 'low' if 'row_count' in s else ('high' if 'error_rate' in s else 'other'))

# 4) Mark anomalies per rules:
# high: value >= p95
# low: value <= p05
def mark_anom(row):
    v = row['value']
    if pd.isna(v):
        return False
    if row['rule'] == 'high':
        if pd.isna(row['p95']):
            return False
        return v >= row['p95']
    if row['rule'] == 'low':
        if pd.isna(row['p05']):
            return False
        return v <= row['p05']
    return False

trace_win['is_anom'] = trace_win.apply(mark_anom, axis=1)

# Compute extremal per group: max for 'high', min for 'low'
ext_high = (
    trace_win[trace_win['rule']=='high']
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .max()
    .rename(columns={'value':'extremal_value_in_window'})
)
ext_low = (
    trace_win[trace_win['rule']=='low']
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .min()
    .rename(columns={'value':'extremal_value_in_window'})
)
extrema = pd.concat([ext_high, ext_low], ignore_index=True)

# anomaly_count and earliest_anom_ts per group
anom_agg = (
    trace_win[trace_win['is_anom']]
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(anomaly_count=('is_anom','sum'), earliest_anom_ts=('dt','min'))
)

# Combine with thresholds and extrema
groups = thr.merge(extrema, on=['cmdb_id','trace_name'], how='left')
groups = groups.merge(anom_agg, on=['cmdb_id','trace_name'], how='left')

groups['anomaly_count'] = groups['anomaly_count'].fillna(0).astype(int)
groups['earliest_anom_ts'] = groups['earliest_anom_ts'].dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %Z")

# Compute threshold_used and severity per group
def compute_thr_sev(row):
    rule = row['trace_name'].lower()
    extremal = row.get('extremal_value_in_window', np.nan)
    p95 = row.get('p95', np.nan)
    p05 = row.get('p05', np.nan)
    thr_val = np.nan
    sev = np.nan
    if 'error_rate' in rule:
        thr_val = p95
        if pd.isna(extremal):
            sev = np.nan
        else:
            if pd.isna(p95) or p95 == 0:
                base = 0 if pd.isna(p95) else p95
                sev = abs(extremal - base)
            else:
                sev = (extremal - p95) / p95
    elif 'row_count' in rule:
        thr_val = p05
        if pd.isna(extremal):
            sev = np.nan
        else:
            if pd.isna(p05) or p05 == 0:
                base = 0 if pd.isna(p05) else p05
                sev = abs(base - extremal)
            else:
                sev = (p05 - extremal) / p05
    return pd.Series({'threshold_used': thr_val, 'severity': sev})

thrsev = groups.apply(compute_thr_sev, axis=1)
groups = pd.concat([groups, thrsev], axis=1)

# Prepare final output columns
out = groups[['cmdb_id','trace_name','anomaly_count','earliest_anom_ts','extremal_value_in_window','threshold_used','severity']].copy()

# 5) Top 20 by severity desc (include ties)
out_sorted = out.sort_values(by=['severity','anomaly_count'], ascending=[False, False]).reset_index(drop=True)
if len(out_sorted) <= 20:
    top20 = out_sorted
else:
    cutoff = out_sorted.iloc[19]['severity']
    top20 = out_sorted[out_sorted['severity'] >= cutoff].reset_index(drop=True)

# Total number of groups with anomaly_count > 0 within this restricted set
total_groups_with_anom = int((out['anomaly_count'] > 0).sum())

# Compact display
top20.head(20), total_groups_with_anom

```
Out[8]:
```
Summary (trace analysis limited to cmdb_ids that had "container.node-6" KPIs; traces filtered to only error_rate and row_count; window 2022-03-20 14:00–14:30 Asia/Shanghai):

What I measured
- Used global thresholds (P95/P05 computed from full series) and flagged:
  - error_rate points as anomalous when value >= P95
  - row_count points as anomalous when value <= P05
- Focused only on cmdb_ids that had any 'container.node-6' KPI.

Top 20 groups by severity (highest first) — examples (compact):
1. productcatalogservice-0 | trace.from_checkoutservice-2.row_count
   - anomaly_count: 16, earliest_anom_ts: 2022-03-20 14:01:00 CST
   - extremal_value_in_window: 1.0, threshold_used (P05): 2.0, severity: 0.50
2. productcatalogservice-1 | trace.from_checkoutservice-2.row_count
   - anomaly_count: 15, earliest_anom_ts: 2022-03-20 14:02:00 CST
   - extremal: 1.0, P05: 2.0, severity: 0.50
3. shippingservice-0 | trace.from_checkoutservice-2.row_count
   - anomaly_count: 15, earliest: 14:01, extremal: 1.0, P05: 2.0, severity: 0.50
4. shippingservice-1 | trace.from_checkoutservice-2.row_count
   - anomaly_count: 15, earliest: 14:00, extremal: 1.0, P05: 2.0, severity: 0.50
5. cartservice-1 | trace.from_checkoutservice-0.row_count
   - anomaly_count: 13, earliest: 14:03, extremal: 1.0, P05: 2.0, severity: 0.50
(others in the top 20 include many row_count drops for checkout/cart/shipping/payment paths; several groups show identical severity = 0.5 from extremal=1 vs P05=2, and some groups have extremal=2 vs P05=4 with same relative severity.)

Aggregate count
- Total distinct cmdb_id / trace_name groups (within this restricted set) with anomaly_count > 0: 454 groups.

Interpretation and recommended focus
- The top-ranked anomalies are mainly row_count drops (functional impact): many downstream service row_counts from checkout → cart / checkout → shipping decreased to ~50% of their normal low-percentile (P05), indicating fewer successful/recorded calls in the window.
- Earliest anomalous timestamps for top groups occur between 14:00 and ~14:05, often starting at 14:01–14:03, suggesting an incident that propagated quickly after window start.
- Prioritize investigation on the checkout → cart → shipping call paths and the listed services (productcatalogservice, shippingservice, cartservice, checkoutservice, paymentservice) because they show the largest relative reductions in row_count (severity = 0.5).
- Because many thresholds are small integers, also check whether instrumentation/metrics sampling changed (false positives) versus real functional failures (request drops/timeouts). Correlate these trace drops with the high-severity container.node-6 KPIs you previously found (e.g., network, IO, or file-descriptor spikes).

If you want, I can:
- Return the full top-200 table sorted by severity,
- Or re-rank these groups by absolute anomaly_count or earliest start time for triage.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                               trace_name  anomaly_count         earliest_anom_ts  extremal_value_in_window  threshold_used  severity
0   productcatalogservice-0   trace.from_checkoutservice-2.row_count             16  2022-03-20 14:01:00 CST                       1.0             2.0       0.5
1   productcatalogservice-1   trace.from_checkoutservice-2.row_count             15  2022-03-20 14:02:00 CST                       1.0             2.0       0.5
2         shippingservice-0   trace.from_checkoutservice-2.row_count             15  2022-03-20 14:01:00 CST                       1.0             2.0       0.5
3         shippingservice-1   trace.from_checkoutservice-2.row_count             15  2022-03-20 14:00:00 CST                       1.0             2.0       0.5
4             cartservice-1   trace.from_checkoutservice-0.row_count             13  2022-03-20 14:03:00 CST                       1.0             2.0       0.5
5         checkoutservice-0         trace.to_cartservice-1.row_count             13  2022-03-20 14:03:00 CST                       1.0             2.0       0.5
6         checkoutservice-0     trace.to_shippingservice-1.row_count             13  2022-03-20 14:05:00 CST                       1.0             2.0       0.5
7         shippingservice-1   trace.from_checkoutservice-0.row_count             13  2022-03-20 14:05:00 CST                       1.0             2.0       0.5
8         checkoutservice-1     trace.to_shippingservice-1.row_count             11  2022-03-20 14:00:00 CST                       1.0             2.0       0.5
9         shippingservice-1   trace.from_checkoutservice-1.row_count             11  2022-03-20 14:00:00 CST                       1.0             2.0       0.5
10       checkoutservice2-0         trace.from_frontend2-0.row_count              7  2022-03-20 14:03:00 CST                       1.0             2.0       0.5
11       checkoutservice2-0        trace.to_cartservice2-0.row_count              7  2022-03-20 14:03:00 CST                       2.0             4.0       0.5
12       checkoutservice2-0       trace.to_emailservice2-0.row_count              7  2022-03-20 14:03:00 CST                       1.0             2.0       0.5
13       checkoutservice2-0     trace.to_paymentservice2-0.row_count              7  2022-03-20 14:03:00 CST                       1.0             2.0       0.5
14       checkoutservice2-0    trace.to_shippingservice2-0.row_count              7  2022-03-20 14:03:00 CST                       2.0             4.0       0.5
15          emailservice2-0  trace.from_checkoutservice2-0.row_count              7  2022-03-20 14:03:00 CST                       1.0             2.0       0.5
16              frontend2-0    trace.to_checkoutservice2-0.row_count              7  2022-03-20 14:03:00 CST                       1.0             2.0       0.5
17        paymentservice2-0  trace.from_checkoutservice2-0.row_count              7  2022-03-20 14:03:00 CST                       1.0             2.0       0.5
18       shippingservice2-0  trace.from_checkoutservice2-0.row_count              7  2022-03-20 14:03:00 CST                       2.0             4.0       0.5
19        checkoutservice-0     trace.to_currencyservice-1.row_count              6  2022-03-20 14:05:00 CST                       1.0             2.0       0.5, 454)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing 'log' and 'tz' if available, otherwise load/define
try:
    log
except NameError:
    log = pd.read_csv("log.csv")
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure columns and types
log['value'] = pd.to_numeric(log['value'], errors='coerce')
log['dt'] = pd.to_datetime(log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Incident window (inclusive)
start_ts = pd.Timestamp("2022-03-20 14:00:00", tz=tz)
end_ts = pd.Timestamp("2022-03-20 14:30:00", tz=tz)

# 1) Filter to window and log_name == 'log.error_count'
log_win = log[(log['dt'] >= start_ts) & (log['dt'] <= end_ts) & (log['log_name'] == 'log.error_count')].copy()

# 2) Aggregates per cmdb_id:
# total_error_sum = sum(value)
# minutes_with_error = count of distinct timestamps where value>0
# earliest_error_ts = min timestamp where value>0
if log_win.shape[0] == 0:
    output = "no error_count > 0 in log.csv within window"
else:
    # Compute total_error_sum per cmdb_id
    total_sum = log_win.groupby('cmdb_id', as_index=False)['value'].sum().rename(columns={'value':'total_error_sum'})
    # minutes_with_error: count distinct dt (minutes) where value>0
    positive = log_win[log_win['value'] > 0].copy()
    if positive.shape[0] == 0:
        # No positive values
        output = "no error_count > 0 in log.csv within window"
    else:
        minutes_with_error = positive.groupby('cmdb_id', as_index=False).agg(minutes_with_error=('dt', lambda s: s.dt.floor('T').nunique()))
        earliest = positive.groupby('cmdb_id', as_index=False).agg(earliest_error_ts=('dt', 'min'))
        # merge
        agg = total_sum.merge(minutes_with_error, on='cmdb_id', how='left').merge(earliest, on='cmdb_id', how='left')
        # format earliest_error_ts
        agg['earliest_error_ts'] = agg['earliest_error_ts'].dt.strftime("%Y-%m-%d %H:%M:%S %Z")
        # keep only cmdb_ids with minutes_with_error > 0 (should be implied)
        agg = agg[agg['minutes_with_error'].fillna(0) > 0].copy()
        # sort by total_error_sum desc and limit top 20
        agg = agg.sort_values(by='total_error_sum', ascending=False).reset_index(drop=True)
        result = agg[['cmdb_id','total_error_sum','minutes_with_error','earliest_error_ts']].head(20)
        output = result

output

```
Out[9]:
```
I filtered log.csv to 2022-03-20 14:00–14:30 (Asia/Shanghai) and to log_name == "log.error_count". There were no entries with value > 0 in that window — i.e., no error_count observed in logs during the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

no error_count > 0 in log.csv within window```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse existing trace and cn6_cmdb_list_full if present
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

try:
    cn6_cmdb_list_full
except NameError:
    # derive from metric if necessary
    try:
        metric
    except NameError:
        metric = pd.read_csv("metric.csv")
    cn6_cmdb_list_full = sorted(metric[metric['kpi_name'].astype(str).str.contains('container.node-6', na=False)]['cmdb_id'].dropna().unique().tolist())

tz = pytz.timezone('Asia/Shanghai')

# Ensure types and timezone-aware datetimes
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
trace['dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Restrict to cmdb_ids of interest and trace_name containing 'row_count'
mask_cmdb = trace['cmdb_id'].isin(cn6_cmdb_list_full)
mask_rowcount = trace['trace_name'].astype(str).str.lower().str.contains('row_count') | trace['trace_name'].astype(str).str.lower().str.contains('rowcount')
trace_row = trace[mask_cmdb & mask_rowcount].copy()

# 1b) Compute global P05 per (cmdb_id, trace_name) using full series (before window filtering)
p05_df = (
    trace_row.dropna(subset=['value'])
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .quantile(0.05)
    .rename(columns={'value':'p05'})
)

# 2) Filter to incident window (inclusive)
start_ts = pd.Timestamp("2022-03-20 14:00:00", tz=tz)
end_ts = pd.Timestamp("2022-03-20 14:30:00", tz=tz)
trace_win = trace_row[(trace_row['dt'] >= start_ts) & (trace_row['dt'] <= end_ts)].copy()

# Merge p05 into window rows
trace_win = trace_win.merge(p05_df, on=['cmdb_id','trace_name'], how='left')

# Determine anomalies per-row: for row_count anomalous when value <= p05 (handling p05==0)
trace_win['is_anom'] = trace_win.apply(
    lambda r: (False if pd.isna(r['value']) or pd.isna(r['p05']) else (r['value'] <= r['p05'])), axis=1
)

# Per (cmdb_id, trace_name): compute anomaly_count and earliest_anom_ts and min_in_window
group_anom = (
    trace_win[trace_win['is_anom']]
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(anomaly_count=('is_anom','sum'), earliest_anom_ts=('dt','min'), min_in_window=('value','min'))
)

# Merge with p05 to ensure groups with thresholds included
group_stats = p05_df.merge(group_anom, on=['cmdb_id','trace_name'], how='left')
group_stats['anomaly_count'] = group_stats['anomaly_count'].fillna(0).astype(int)
group_stats['earliest_anom_ts'] = group_stats['earliest_anom_ts'].dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %Z")
group_stats['min_in_window'] = group_stats['min_in_window'] = group_stats['min_in_window'] = group_stats.get('min_in_window', group_stats.get('min_in_window', np.nan))

# Compute per-(cmdb_id,trace_name) severity = (p05 - min_in_window)/p05 (if p05==0 use abs(p05 - min))
def compute_sev_row(row):
    p05 = row['p05']
    minv = row['min_in_window']
    if pd.isna(minv):
        return np.nan
    if pd.isna(p05) or p05 == 0:
        base = 0 if pd.isna(p05) else p05
        return abs(base - minv)
    else:
        return (p05 - minv) / p05

group_stats['severity_per_trace'] = group_stats.apply(compute_sev_row, axis=1)

# 2) Derive per-pod (cmdb_id) aggregation:
# - pod_has_anom: any anomaly_count>0 across its trace_names
# - pod_earliest_anom_ts: min earliest_anom_ts among its anomalous trace_names
# - pod_severity: max severity_per_trace among its anomalous trace_names
# - pod_anom_count_total: sum of anomaly_count across its trace_names
pod_agg = (
    group_stats[group_stats['anomaly_count'] > 0]
    .groupby('cmdb_id', as_index=False)
    .agg(
        pod_anomalous_trace_count=('trace_name','nunique'),
        pod_earliest_anom_ts=('earliest_anom_ts','min'),
        pod_max_severity=('severity_per_trace','max'),
        pod_anomaly_count_total=('anomaly_count','sum')
    )
)

# Format pod_earliest_anom_ts (already string) and ensure types
pod_agg['pod_earliest_anom_ts'] = pod_agg['pod_earliest_anom_ts']

# 3) Map pod cmdb_id to service base name by removing trailing '-<digits>'
def map_service(name):
    if pd.isna(name):
        return name
    return re.sub(r'-\d+$', '', name)

pod_agg['service_name'] = pod_agg['cmdb_id'].apply(map_service)

# 4) Focus services set
focus_services = ['checkoutservice','cartservice','shippingservice','productcatalogservice','paymentservice','recommendationservice']

# For each service, compute aggregates among pods where service_name == target
service_rows = []
for svc in focus_services:
    df = pod_agg[pod_agg['service_name'] == svc]
    service_pod_count = int(df.shape[0])
    if service_pod_count == 0:
        service_earliest_anom = pd.NA
        service_max_severity = np.nan
        total_anomalous_pod_rowcounts = 0
    else:
        # earliest among pods
        # pod_earliest_anom_ts stored as string; convert back to datetime for min
        dt_series = pd.to_datetime(df['pod_earliest_anom_ts'], utc=True).dt.tz_convert(tz)
        service_earliest_anom = dt_series.min().strftime("%Y-%m-%d %H:%M:%S %Z")
        service_max_severity = float(df['pod_max_severity'].max())
        total_anomalous_pod_rowcounts = int(df['pod_anomaly_count_total'].sum())
    service_rows.append({
        'service_name': svc,
        'service_pod_count': service_pod_count,
        'service_earliest_anom': service_earliest_anom,
        'service_max_severity': service_max_severity,
        'total_anomalous_pod_rowcounts': total_anomalous_pod_rowcounts
    })

service_table = pd.DataFrame(service_rows, columns=['service_name','service_pod_count','service_earliest_anom','service_max_severity','total_anomalous_pod_rowcounts'])

# 5) Total number of distinct pods (across all services) that had any row_count anomaly
total_pods_with_anom = int(pod_agg['cmdb_id'].nunique())

# Compact outputs
service_table, total_pods_with_anom

```
Out[10]:
```
Summary (row_count anomalies, window 2022-03-20 14:00–14:30 Asia/Shanghai):

Service-level summary (service_name — service_pod_count — service_earliest_anom — service_max_severity — total_anomalous_pod_rowcounts)
- checkoutservice — 2 pods — 2022-03-20 22:00:00 CST — 0.50 — 388
- cartservice — 3 pods — 2022-03-20 22:00:00 CST — 0.50 — 120
- shippingservice — 2 pods — 2022-03-20 22:00:00 CST — 0.50 — 93
- productcatalogservice — 3 pods — 2022-03-20 22:00:00 CST — 0.50 — 161
- paymentservice — 3 pods — 2022-03-20 22:00:00 CST — 0.00 — 161
- recommendationservice — 3 pods — 2022-03-20 22:01:00 CST — 0.0588 — 109

Total distinct pods across these services with any row_count anomaly in the window: 34

Interpretation (brief): multiple pods across checkout/cart/shipping/productcatalog show row_count drops (severity ≈ 0.5), indicating notable functional impact on those call paths. Recommendation: prioritize investigating checkout → cart → shipping flows and related pods.

The original code execution output of IPython Kernel is also provided below for reference:

(            service_name  service_pod_count    service_earliest_anom  service_max_severity  total_anomalous_pod_rowcounts
0        checkoutservice                  2  2022-03-20 22:00:00 CST              0.500000                            388
1            cartservice                  3  2022-03-20 22:00:00 CST              0.500000                            120
2        shippingservice                  2  2022-03-20 22:00:00 CST              0.500000                             93
3  productcatalogservice                  3  2022-03-20 22:00:00 CST              0.500000                            161
4         paymentservice                  3  2022-03-20 22:00:00 CST              0.000000                            161
5  recommendationservice                  3  2022-03-20 22:01:00 CST              0.058824                            109, 34)```
```